# Step 3 — Training

Reads the step-2 cache plus whichever label file `LABELS` points at, so swapping in
revised pseudo-labels is a one-line change.

**Setup:** attach (1) the step-2 cache notebook output, (2) your labels dataset,
(3) the competition. Accelerator **GPU**, Internet **ON** (pretrained weights only —
the saved checkpoint is self-contained, so the offline submission notebook is fine).

**Single GPU on purpose.** `nn.DataParallel` was measured at **8.72 s/step vs 0.66 s
single-GPU on the same batch** — 28 img/s against 364. It re-replicates the module and
broadcasts every BatchNorm buffer across PCIe on each forward, and Kaggle's two T4s
have no NVLink. Don't re-enable it. If you ever want both GPUs, use DDP via
`torchrun`, never DataParallel.

**Architecture.** Each plane's slices go through a 2D backbone as overlapping 3-slice
stacks → attention-pool over slices → **per-label** attention-pool over planes → 12
logits. Per-label is what lets PF OA look at axial while ACL looks at sagittal.

## 0 · Config

In [1]:
1

1

In [2]:
ls

__notebook__.ipynb


In [3]:
cd knee-mri-step2b-cache6


[Errno 2] No such file or directory: 'knee-mri-step2b-cache6'
/kaggle/working


In [4]:
import os, gc, math, time, json, random, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

# ---- inputs ------------------------------------------------------------------
CACHE_DIR = "/kaggle/input/notebooks/shreyanshgoyaliitb/knee-mri-step2b-cache6"
LABELS    = "/kaggle/input/datasets/shreyanshgoyaliitb/pseudo-labels/pseudo_labels.csv"
COMP_DIR  = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
OUT       = "/kaggle/working"
# -----------------------------------------------------------------------------

LABEL_COLS = ["ACL","MCL","Medial Meniscus","Lateral Meniscus","Medial OA","Lateral OA",
              "PF OA","Effusion","Synovitis","Baker's","Contusion","Fracture"]

_slots = json.load(open(os.path.join(CACHE_DIR, "slots.json")))
PLANES   = _slots["slots"]          
print("slots:", PLANES)

BACKBONE    = "tf_efficientnet_b0.ns_jft_in1k"
IMG         = 224
N_SLICES    = 16       # overridden from the cache below
T_TRAIN     = 10       # slices sampled per plane per step (augmentation + speed)
BATCH       = 4        # one T4 saturates here: batch 8 gave 364 img/s vs 369
EPOCHS      = 8
LR          = 3e-4
WD          = 1e-2
SEED        = 42
NUM_WORKERS = 4        # Kaggle gives 4 cores regardless of accelerator

DEV = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
torch.backends.cudnn.benchmark = True          # shapes are fixed -> let cuDNN autotune

print("device:", DEV, torch.cuda.get_device_name(0) if DEV == "cuda" else "")
if DEV == "cuda" and torch.cuda.device_count() > 1:
    print(f"note: {torch.cuda.device_count()} GPUs visible, using 1 by design "
          f"(DataParallel benchmarked 13x slower here)")

slots: ['SAG_FLUID_FS', 'COR_FLUID_FS', 'AX_FLUID_FS', 'SAG_FLUID_NOFS', 'COR_T1', 'SAG_T1']
device: cuda Tesla T4
note: 2 GPUs visible, using 1 by design (DataParallel benchmarked 13x slower here)


## 1 · Data

In [5]:
index = pd.read_csv(os.path.join(CACHE_DIR, "cache_index.csv"))
cache = np.load(os.path.join(CACHE_DIR, "cache.npy"), mmap_mode="r")
print("cache", cache.shape, cache.dtype, f"{cache.nbytes/1e9:.1f} GB")

# Geometry comes from the cache, not the constants -- rebuilding the cache at a
# different depth must never silently corrupt training.
_, N_PLANES, N_SLICES, H_C, W_C = cache.shape
assert H_C == W_C, "non-square cache"
if (H_C, N_PLANES) != (IMG, len(PLANES)):
    print(f"  overriding config: IMG {IMG}->{H_C}, planes {len(PLANES)}->{N_PLANES}")
IMG = H_C
T_TRAIN = min(T_TRAIN, N_SLICES)
print(f"  geometry: {N_PLANES} planes x {N_SLICES} slices x {IMG}px | T_TRAIN={T_TRAIN}")

# Labels: accept train_targets.csv (has 'source') or pseudo_labels.csv (doesn't).
targets = pd.read_csv(LABELS)
print(f"\nlabels: {os.path.basename(LABELS)}  {targets.shape}")

if "source" not in targets.columns:
    print("  no 'source' column -> pseudo-only file; recovering gold rows from train.csv")
    tr = pd.read_csv(os.path.join(COMP_DIR, "train.csv"))
    for c in LABEL_COLS:
        tr[c] = pd.to_numeric(tr.get(c), errors="coerce")
    gold = tr.loc[tr[LABEL_COLS].notna().all(axis=1), ["StudyInstanceUID", *LABEL_COLS]].copy()
    gold["source"] = "gold"
    ps = targets[["StudyInstanceUID", *LABEL_COLS]].copy()
    ps["source"] = "pseudo"
    ps = ps[~ps["StudyInstanceUID"].isin(set(gold["StudyInstanceUID"]))]
    targets = pd.concat([gold, ps], ignore_index=True)
    print(f"  gold {len(gold)} + pseudo {len(ps)} = {len(targets)}")

targets[LABEL_COLS] = targets[LABEL_COLS].astype("float32")
assert targets["StudyInstanceUID"].is_unique, "duplicate studies in the label file"

df = index[index["n_present"] > 0].merge(
        targets[["StudyInstanceUID", *LABEL_COLS, "source"]],
        on="StudyInstanceUID", how="inner")
print(f"\nusable studies: {len(df):,}  ({df['source'].value_counts().to_dict()})")

y_all = df[LABEL_COLS].to_numpy(np.float32)
print(f"observed label cells: {(~np.isnan(y_all)).mean():.1%}")
print("\npositive rate (observed cells only):")
print(pd.Series(np.nanmean(y_all, axis=0), index=LABEL_COLS).round(3).to_string())

cache (4407, 6, 12, 224, 224) uint8 15.9 GB
  geometry: 6 planes x 12 slices x 224px | T_TRAIN=10

labels: pseudo_labels.csv  (4349, 14)
  no 'source' column -> pseudo-only file; recovering gold rows from train.csv
  gold 58 + pseudo 4349 = 4407

usable studies: 4,407  ({'pseudo': 4349, 'gold': 58})
observed label cells: 100.0%

positive rate (observed cells only):
ACL                 0.165
MCL                 0.140
Medial Meniscus     0.390
Lateral Meniscus    0.145
Medial OA           0.361
Lateral OA          0.223
PF OA               0.446
Effusion            0.581
Synovitis           0.231
Baker's             0.250
Contusion           0.149
Fracture            0.071


In [6]:
# The gold studies are the only human labels -> held out, never trained on.
# A slice of pseudo studies gives a larger but noisier second signal.
gold_mask = df["source"].eq("gold").to_numpy()
rng = np.random.default_rng(SEED)
pseudo_idx = np.where(~gold_mask)[0]
rng.shuffle(pseudo_idx)
n_val = int(0.08 * len(pseudo_idx))

val_pseudo = pseudo_idx[:n_val]
train_idx  = pseudo_idx[n_val:]
val_gold   = np.where(gold_mask)[0]

print(f"train        {len(train_idx):,} (pseudo only)")
print(f"val pseudo   {len(val_pseudo):,}")
print(f"val gold     {len(val_gold):,}  <- the number that matters")

train        4,002 (pseudo only)
val pseudo   347
val gold     58  <- the number that matters


In [7]:
import cv2

class KneeDS(Dataset):
    def __init__(self, df, rows, train):
        self.df = df.reset_index(drop=True)
        self.rows = rows
        self.train = train
        self.y = self.df[LABEL_COLS].to_numpy(np.float32)
        self.present = self.df[[f"present_{s}" for s in range(len(PLANES))]].to_numpy(bool)
        self.row_of = self.df["row"].to_numpy(int)
        self.cache = None                      # memmap opened lazily, per worker

    def __len__(self):
        return len(self.rows)

    def _aug(self, vol):
        # NO horizontal or vertical flip: a horizontal flip in the coronal plane swaps
        # medial and lateral, which would invert four of the twelve labels.
        h, w = vol.shape[-2:]
        ang = np.random.uniform(-12, 12)
        sc  = np.random.uniform(0.9, 1.1)
        M = cv2.getRotationMatrix2D((w/2, h/2), ang, sc)
        M[:, 2] += np.random.uniform(-0.06, 0.06, 2) * w
        out = np.empty_like(vol)
        for i in range(vol.shape[0]):
            out[i] = cv2.warpAffine(vol[i], M, (w, h), flags=cv2.INTER_LINEAR,
                                    borderValue=0)
        if np.random.rand() < 0.5:
            out = np.clip(out.astype(np.float32) * np.random.uniform(0.85, 1.15)
                          + np.random.uniform(-15, 15), 0, 255).astype(np.uint8)
        return out

    def __getitem__(self, i):
        if self.cache is None:
            self.cache = np.load(os.path.join(CACHE_DIR, "cache.npy"), mmap_mode="r")
        j = self.rows[i]
        r = self.row_of[j]
        present = self.present[j]

        if self.train:
            # Slice the memmap BEFORE materialising, so the discarded slices are
            # never read off disk at all (~37% less I/O per sample).
            avail = self.cache.shape[2]
            t = min(T_TRAIN, avail)
            t0 = np.random.randint(0, avail - t + 1)
            vol = np.asarray(self.cache[r][:, t0:t0 + t])
            vol = np.stack([self._aug(v) if present[s] else v
                            for s, v in enumerate(vol)])
        else:
            vol = np.asarray(self.cache[r])          # eval uses every slice

        return (torch.from_numpy(np.ascontiguousarray(vol)),
                torch.from_numpy(present.copy()),
                torch.from_numpy(self.y[j]))


def collate(batch):
    xs, ps, ys = zip(*batch)
    return torch.stack(xs), torch.stack(ps), torch.stack(ys)


ds_tr = KneeDS(df, train_idx, True)
ds_vp = KneeDS(df, val_pseudo, False)
ds_vg = KneeDS(df, val_gold, False)

x, p, y = ds_tr[0]
print("train sample:", tuple(x.shape), x.dtype, "| present", p.tolist())
x, p, y = ds_vg[0]
print("eval  sample:", tuple(x.shape), x.dtype)

train sample: (6, 10, 224, 224) torch.uint8 | present [True, True, True, False, False, True]
eval  sample: (6, 12, 224, 224) torch.uint8


## 2 · Model

`make_neighbours` gives each slice a 3-channel view of its neighbours, so an
ImageNet-pretrained 2D backbone sees through-plane context for free.

`LabelAttnPool` gives **each finding its own weighting over planes**. With a single
shared attention the axial plane collapsed to 0.001 — nine of twelve findings live in
sagittal and coronal, so the shared scorer starved the minority, and PF OA and
Synovitis (the axial-dependent ones) were the two weakest columns.

In [8]:
import timm

def make_neighbours(x):
    # (B,S,T,H,W) -> (B,S,T,3,H,W) from slices t-1,t,t+1 (edges clamped)
    prev = torch.cat([x[:, :, :1], x[:, :, :-1]], dim=2)
    nxt  = torch.cat([x[:, :, 1:], x[:, :, -1:]], dim=2)
    return torch.stack([prev, x, nxt], dim=3)


class AttnPool(nn.Module):
    # Attention over slices within one plane.
    def __init__(self, dim, hidden=128):
        super().__init__()
        self.score = nn.Sequential(nn.Linear(dim, hidden), nn.Tanh(), nn.Linear(hidden, 1))

    def forward(self, x, mask=None):               # x (B,T,D)
        a = self.score(x).squeeze(-1)
        if mask is not None:
            a = a.masked_fill(~mask, torch.finfo(a.dtype).min)
        a = a.softmax(dim=-1)
        return (x * a.unsqueeze(-1)).sum(1), a


class LabelAttnPool(nn.Module):
    # Per-label attention over planes: each finding picks the view it needs.
    def __init__(self, dim, n_labels):
        super().__init__()
        self.score = nn.Linear(dim, n_labels)

    def forward(self, x, mask):                    # x (B,S,D), mask (B,S)
        a = self.score(x)                          # (B,S,L)
        a = a.masked_fill(~mask.unsqueeze(-1), torch.finfo(a.dtype).min)
        a = a.softmax(dim=1)                       # over planes, per label
        z = torch.einsum("bsd,bsl->bld", x, a)     # (B,L,D)
        return z, a


class KneeNet(nn.Module):
    def __init__(self, backbone=BACKBONE, n_out=len(LABEL_COLS), n_planes=len(PLANES),
                 pretrained=True, drop=0.3):
        super().__init__()
        self.enc = timm.create_model(backbone, pretrained=pretrained,
                                     num_classes=0, in_chans=3)
        d = self.enc.num_features
        self.slice_pool = AttnPool(d)
        self.plane_emb = nn.Parameter(torch.zeros(n_planes, d))
        self.plane_pool = LabelAttnPool(d, n_out)
        self.drop = nn.Dropout(drop)
        self.head_w = nn.Parameter(torch.zeros(n_out, d))
        self.head_b = nn.Parameter(torch.zeros(n_out))
        nn.init.trunc_normal_(self.head_w, std=0.02)

    def forward(self, x, present):
        # autocast entered HERE, not in the training loop: it is thread-local, so a
        # loop-level autocast never reaches a threaded forward. Verified reaching the
        # encoder as float16.
        with torch.amp.autocast("cuda", enabled=x.is_cuda):
            B, S, T, H, W = x.shape
            x = x.float().div_(255.).sub_(0.449).div_(0.226)   # ImageNet grey stats
            x3 = make_neighbours(x).reshape(B * S * T, 3, H, W)

            f = self.enc(x3).reshape(B * S, T, -1)
            v, _ = self.slice_pool(f)
            v = v.reshape(B, S, -1) + self.plane_emb.unsqueeze(0)

            m = present.clone()
            m[~m.any(dim=1)] = True                # guard: never all-masked
            z, plane_att = self.plane_pool(v, m)
            z = self.drop(z)
            logits = (z * self.head_w.unsqueeze(0)).sum(-1) + self.head_b
        return logits, plane_att


model = KneeNet().to(DEV)
print(f"{BACKBONE} | {sum(p.numel() for p in model.parameters())/1e6:.1f}M params "
      f"| feature dim {model.enc.num_features}")

model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

tf_efficientnet_b0.ns_jft_in1k | 4.2M params | feature dim 1280


## 3 · Loss and metric

In [9]:
def masked_bce(logits, targets):
    # BCE over (B,12) targets that may contain NaN. NaN cells get exactly zero
    # gradient, so a study the extractor abstained on still trains the other heads.
    mask = ~torch.isnan(targets)
    if not mask.any():
        return logits.sum() * 0.0
    loss = F.binary_cross_entropy_with_logits(
        logits, torch.nan_to_num(targets, nan=0.0), reduction="none")
    return (loss * mask).sum() / mask.sum()


from sklearn.metrics import roc_auc_score

def macro_auc(y, s):
    aucs = {}
    for k, c in enumerate(LABEL_COLS):
        yy, ss = y[:, k], s[:, k]
        ok = ~np.isnan(yy) & ~np.isnan(ss)
        yb = (yy[ok] >= 0.5).astype(int)           # targets can be fractional
        aucs[c] = (roc_auc_score(yb, ss[ok])
                   if ok.sum() > 1 and 0 < yb.sum() < len(yb) else np.nan)
    return float(np.nanmean(list(aucs.values()))), aucs

## 4 · Train

In [10]:
dl_tr = DataLoader(ds_tr, batch_size=BATCH, shuffle=True, num_workers=NUM_WORKERS,
                   collate_fn=collate, pin_memory=True, drop_last=True,
                   persistent_workers=NUM_WORKERS > 0, prefetch_factor=4 if NUM_WORKERS else None)
dl_vp = DataLoader(ds_vp, batch_size=BATCH, shuffle=False, num_workers=NUM_WORKERS,
                   collate_fn=collate, persistent_workers=NUM_WORKERS > 0)
dl_vg = DataLoader(ds_vg, batch_size=BATCH, shuffle=False, num_workers=NUM_WORKERS,
                   collate_fn=collate, persistent_workers=NUM_WORKERS > 0)

opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
steps = EPOCHS * len(dl_tr)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=steps, pct_start=0.1)
scaler = torch.amp.GradScaler(DEV, enabled=(DEV == "cuda"))
print(f"{len(dl_tr)} steps/epoch x {EPOCHS} epochs = {steps:,} steps")


@torch.no_grad()
def evaluate(dl, desc="eval"):
    model.eval()
    Y, S = [], []
    for x, p, y in tqdm(dl, desc=desc, leave=False, mininterval=2.0, unit="batch"):
        x, p = x.to(DEV, non_blocking=True), p.to(DEV)
        logit, _ = model(x, p)
        S.append(torch.sigmoid(logit.float()).cpu().numpy())
        Y.append(y.numpy())
    return np.concatenate(Y), np.concatenate(S)

1000 steps/epoch x 8 epochs = 8,000 steps


In [11]:
best, hist = -1.0, []
epoch_bar = tqdm(range(EPOCHS), desc="training", unit="epoch", position=0)

for ep in epoch_bar:
    model.train()
    t0, run = time.time(), 0.0
    step_bar = tqdm(dl_tr, desc=f"  epoch {ep}", leave=False, position=1,
                    mininterval=2.0, unit="batch")

    for i, (x, p, y) in enumerate(step_bar, 1):
        x, p, y = x.to(DEV, non_blocking=True), p.to(DEV), y.to(DEV)
        logit, _ = model(x, p)                     # autocast lives inside forward
        loss = masked_bce(logit.float(), y)

        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        scaler.step(opt); scaler.update(); sched.step()

        run += loss.item()
        step_bar.set_postfix(loss=f"{run/i:.4f}", lr=f"{sched.get_last_lr()[0]:.1e}")
    step_bar.close()

    yp, sp = evaluate(dl_vp, "  val pseudo"); auc_p, _ = macro_auc(yp, sp)
    yg, sg = evaluate(dl_vg, "  val gold");   auc_g, _ = macro_auc(yg, sg)

    hist.append(dict(epoch=ep, loss=run/len(dl_tr), auc_pseudo=auc_p, auc_gold=auc_g,
                     minutes=(time.time()-t0)/60))
    epoch_bar.set_postfix(gold=f"{auc_g:.4f}", pseudo=f"{auc_p:.4f}", best=f"{max(best,auc_g):.4f}")
    tqdm.write(f"epoch {ep}: loss {run/len(dl_tr):.4f} | AUC gold {auc_g:.4f} "
               f"| AUC pseudo {auc_p:.4f} | {(time.time()-t0)/60:.1f} min")

    if auc_g > best:
        best = auc_g
        sd = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
        torch.save({"model": sd,
                    "cfg": dict(backbone=BACKBONE, img=IMG, n_slices=N_SLICES,
                                planes=PLANES, label_cols=LABEL_COLS, t_train=T_TRAIN),
                    "epoch": ep, "auc_gold": auc_g}, os.path.join(OUT, "best.pt"))
        tqdm.write(f"  saved best.pt (gold AUC {auc_g:.4f})")

epoch_bar.close()
pd.DataFrame(hist).to_csv(os.path.join(OUT, "history.csv"), index=False)
print("\n" + pd.DataFrame(hist).round(4).to_string(index=False))

training:   0%|          | 0/8 [00:00<?, ?epoch/s]

  epoch 0:   0%|          | 0/1000 [00:00<?, ?batch/s]

  val pseudo:   0%|          | 0/87 [00:00<?, ?batch/s]

  val gold:   0%|          | 0/15 [00:00<?, ?batch/s]

epoch 0: loss 0.5111 | AUC gold 0.7656 | AUC pseudo 0.7626 | 13.1 min
  saved best.pt (gold AUC 0.7656)


...

epoch 6: loss 0.2846 | AUC gold 0.8607 | AUC pseudo 0.8553 | 11.9 min


  epoch 7:   0%|          | 0/1000 [00:00<?, ?batch/s]

  val pseudo:   0%|          | 0/87 [00:00<?, ?batch/s]

  val gold:   0%|          | 0/15 [00:00<?, ?batch/s]

epoch 7: loss 0.2689 | AUC gold 0.8647 | AUC pseudo 0.8569 | 12.0 min

 epoch   loss  auc_pseudo  auc_gold  minutes
     0 0.5111      0.7626    0.7656  13.1158
     1 0.4374      0.8173    0.8065  11.8991
     2 0.4022      0.8315    0.8413  11.9646
     3 0.3717      0.8404    0.8480  11.9137
     4 0.3406      0.8557    0.8643  11.9469
     5 0.3088      0.8572    0.8652  11.9373
     6 0.2846      0.8553    0.8607  11.9473
     7 0.2689      0.8569    0.8647  11.9913


## 5 · Per-finding results

In [12]:
1

1

In [13]:
ck = torch.load(os.path.join(OUT, "best.pt"), map_location=DEV, weights_only=False)
(model.module if isinstance(model, nn.DataParallel) else model).load_state_dict(ck["model"])
print(f"loaded epoch {ck['epoch']} | gold AUC {ck['auc_gold']:.4f}\n")

yg, sg = evaluate(dl_vg, "gold");   auc_g, per_g = macro_auc(yg, sg)
yp, sp = evaluate(dl_vp, "pseudo"); auc_p, per_p = macro_auc(yp, sp)

res = pd.DataFrame({
    "gold_AUC":   pd.Series(per_g),
    "pseudo_AUC": pd.Series(per_p),
    "gold_pos":   pd.Series((yg >= 0.5).sum(axis=0).astype(int), index=LABEL_COLS),
}).loc[LABEL_COLS]
print(res.round(3).to_string())
print(f"\nMACRO  gold {auc_g:.4f}   pseudo {auc_p:.4f}")
print("\nGold n=58: per-finding AUC carries roughly +/-0.10 to +/-0.15 here.")
print("Trust the macro figure; treat single-finding differences under 0.1 as noise.")

loaded epoch 5 | gold AUC 0.8652



gold:   0%|          | 0/15 [00:00<?, ?batch/s]

pseudo:   0%|          | 0/87 [00:00<?, ?batch/s]

                  gold_AUC  pseudo_AUC  gold_pos
ACL                  0.949       0.932        24
MCL                  0.855       0.851         9
Medial Meniscus      0.868       0.916        26
Lateral Meniscus     0.725       0.804        23
Medial OA            0.971       0.900        15
Lateral OA           0.867       0.842        11
PF OA                0.816       0.835        21
Effusion             0.889       0.857        35
Synovitis            0.674       0.779        27
Baker's              0.949       0.878        12
Contusion            0.897       0.854        19
Fracture             0.922       0.837        18

MACRO  gold 0.8652   pseudo 0.8572

Gold n=58: per-finding AUC carries roughly +/-0.10 to +/-0.15 here.
Trust the macro figure; treat single-finding differences under 0.1 as noise.


In [14]:
# Which plane does each finding actually use?
atts = []
model.eval()
with torch.no_grad():
    for x, p, y in tqdm(dl_vg, desc="plane attention", leave=False, unit="batch"):
        _, a = model(x.to(DEV), p.to(DEV))
        atts.append(a.float().cpu().numpy())
atts = np.concatenate(atts)                        # (N, planes, labels)

tab = pd.DataFrame(atts.mean(0), index=PLANES, columns=LABEL_COLS).T
tab["uses"] = tab.idxmax(axis=1)
print(tab.round(3).to_string())
print("\nExpected anatomy: ACL and menisci -> Sagittal; MCL and the tibiofemoral")
print("compartments -> Coronal; PF OA, Effusion, Synovitis -> Axial.")
print("If those line up without being told, the model found the anatomy itself.")
print("If one plane is ~0.00 everywhere, the pooling collapsed again.")

plane attention:   0%|          | 0/15 [00:00<?, ?batch/s]

                  SAG_FLUID_FS  COR_FLUID_FS  AX_FLUID_FS  SAG_FLUID_NOFS  COR_T1  SAG_T1          uses
ACL                      0.102         0.625        0.110           0.036   0.094   0.033  COR_FLUID_FS
MCL                      0.068         0.585        0.225           0.030   0.060   0.032  COR_FLUID_FS
Medial Meniscus          0.313         0.479        0.050           0.059   0.056   0.043  COR_FLUID_FS
Lateral Meniscus         0.077         0.688        0.119           0.018   0.085   0.013  COR_FLUID_FS
Medial OA                0.171         0.334        0.111           0.114   0.180   0.090  COR_FLUID_FS
Lateral OA               0.152         0.265        0.174           0.132   0.148   0.129  COR_FLUID_FS
PF OA                    0.210         0.180        0.309           0.118   0.099   0.085   AX_FLUID_FS
Effusion                 0.403         0.180        0.288           0.065   0.036   0.028  SAG_FLUID_FS
Synovitis                0.204         0.349        0.242       

## 6 · Optional: throughput check

Run this if training feels slow. It separates data from compute, which is the only
way to tell them apart — guessing cost a wasted run once already.

In [15]:
import itertools

MB = np.prod(cache.shape[1:]) / 1e6
print(f"{MB:.1f} MB/study | batch {BATCH} -> {MB*BATCH:.1f} MB/step\n")

t0, n = time.time(), 0
for x, p, y in tqdm(itertools.islice(dl_tr, 15), total=15, desc="data only", unit="batch"):
    n += x.shape[0]
dt = time.time() - t0
print(f"DATA only : {dt/15:6.2f} s/batch  ({n*MB/dt:6.1f} MB/s)")

xb, pb, yb = next(iter(dl_tr))
xb, pb, yb = xb.to(DEV), pb.to(DEV), yb.to(DEV)
for _ in range(3):
    lg, _ = model(xb, pb); masked_bce(lg.float(), yb).backward(); model.zero_grad(set_to_none=True)
if DEV == "cuda": torch.cuda.synchronize()

t0 = time.time()
for _ in tqdm(range(10), desc="gpu only", unit="step"):
    lg, _ = model(xb, pb); masked_bce(lg.float(), yb).backward(); model.zero_grad(set_to_none=True)
if DEV == "cuda": torch.cuda.synchronize()
per = (time.time() - t0) / 10
print(f"GPU  only : {per:6.2f} s/batch  ({BATCH*len(PLANES)*T_TRAIN/per:6.0f} img/s)")
print(f"peak GPU memory: {torch.cuda.max_memory_allocated()/1e9:.1f} GB"
      if DEV == "cuda" else "")
print("\nReference on one T4: ~0.33 s/batch, ~370 img/s, ~5.6 GB at batch 4.")

3.6 MB/study | batch 4 -> 14.5 MB/step



data only:   0%|          | 0/15 [00:00<?, ?batch/s]

DATA only :   0.04 s/batch  ( 323.7 MB/s)


gpu only:   0%|          | 0/10 [00:00<?, ?step/s]

GPU  only :   0.61 s/batch  (   395 img/s)
peak GPU memory: 11.1 GB

Reference on one T4: ~0.33 s/batch, ~370 img/s, ~5.6 GB at batch 4.


## 7 · Next

`best.pt` holds weights + config and needs no internet. The submission notebook must:

1. read `test.csv` / `test_series.csv`,
2. run step 2's `load_series()` on the test DICOMs **at inference time** — there is no
   cache for test, and the real test set is swapped in at scoring,
3. assemble the same 3-plane tensor, predict, and write `submission.csv` with the 12
   probability columns in `LABEL_COLS` order.

Confirm `pylibjpeg` is importable offline in the Kaggle image before building it —
that notebook cannot install it.